---
title: "Part 22: Machine Learning with scikit-learn"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/04-ml-intro/04-sklearn-core.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/04-ml-intro/04-sklearn-core.ipynb)

**DS-MLOps Part 4: ML Landscape**

**Python 3.12+ | Author: Anthony Faustine**

## Before you begin

This notebook is **Chapter 4 of Part 4** and translates the theory from Chapters 1-3 into working code.
Before starting, make sure you have completed:

- **Chapter 3 (Part 4):** The ML Workflow and Problem Framing (splits, metrics, bias-variance, learning curves)
- **Part 16:** Type Annotations (applied throughout via type hints)
- **Part 21:** Logging for ML Projects (loguru used at every training step)

You will also need the `modelling` extras installed:

```bash
uv sync --extra modelling
```

The dataset is `university_analytics.csv`, which you have already explored in Parts 1 and 2.
Using a familiar dataset lets you focus on the sklearn API rather than domain understanding.

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).


::: {.callout-note collapse="true" icon=false}
## Topics covered

| Topic | Why it matters |
| --- | --- |
| **sklearn API contract** | Every estimator obeys fit / predict / transform; know the rule once, apply it everywhere |
| **Feature scaling** | Linear models and distance-based models require standardised inputs |
| **Data splits** | Correct train/test separation prevents inflated scores |
| **Baseline models** | DummyRegressor and DummyClassifier set the floor; beat them or go back to the drawing board |
| **LinearRegression and LogisticRegression** | The simplest non-trivial models; understand them deeply before moving to ensembles |
| **Evaluation metrics** | MAE, RMSE, R2 for regression; precision, recall, F1 for classification |
| **Cross-validation** | A distribution of scores is more reliable than one train/val split |
| **Learning curves** | Diagnose underfitting and overfitting visually, as introduced in Chapter 3 |
| **Model persistence** | Save and load trained models with joblib |
| **Type annotations and loguru** | Apply dev-tools best practices to ML code |
:::


::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 22 you will be able to:

| # | Skill | Covered in |
| --- | --- | --- |
| 1 | Describe the sklearn API contract (fit / predict / transform / score) and apply it to any estimator | Sec. 1 |
| 2 | Load and explore a real dataset, selecting numerical features for modelling | Sec. 2 |
| 3 | Apply StandardScaler and MinMaxScaler correctly, fitting only on training data | Sec. 3 |
| 4 | Split data with train_test_split, using stratification for classification | Sec. 4 |
| 5 | Establish a DummyRegressor / DummyClassifier baseline before training any real model | Sec. 5 |
| 6 | Train LinearRegression and LogisticRegression and evaluate with metrics from Chapter 3 | Sec. 6 |
| 7 | Use cross_val_score and StratifiedKFold for reliable model assessment | Sec. 7 |
| 8 | Compute and interpret learning curves, connecting to Chapter 3 Section 5 | Sec. 8 |
| 9 | Persist a trained model with joblib and reload it for inference | Sec. 9 |
| 10 | Build a type-annotated, logged end-to-end training function | Capstone |
:::


## 0. Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import warnings

import joblib
from lets_plot import (
    LetsPlot,
    aes,
    geom_hline,
    geom_line,
    geom_point,
    geom_ribbon,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    scale_fill_manual,
)
from loguru import logger
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    classification_report,
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from ark.plot.theme import modern_theme, pro_colors

warnings.filterwarnings("ignore")

logger.remove()
logger.add(
    lambda msg: print(msg, end=""),
    colorize=False,
    format="{time:HH:mm:ss} | {level:<8} | {message}",
)

LetsPlot.setup_html()

In [ ]:
def _find_data() -> Path:
    candidates = [
        Path("../01-python-basics/data/university_analytics.csv"),
        Path("tutorials/01-python-basics/data/university_analytics.csv"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Cannot locate university_analytics.csv. Run from the repo root or notebook directory.")


DATA_PATH = _find_data()
logger.info(f"Data resolved: {DATA_PATH}")

## 1. The sklearn API Contract

The entire sklearn library is built on one rule:

> **Fit on training data only. Then transform or predict on any split.**

Every class you will ever use inherits from one of two base classes:

- **Transformer** (`StandardScaler`, `MinMaxScaler`, `OneHotEncoder`, ...): calls `fit(X_train)` to learn statistics,
  then `transform(X)` to apply them. The `fit_transform(X)` shortcut does both in one call (use it only on training data).
- **Predictor** (`LinearRegression`, `LogisticRegression`, `RandomForestClassifier`, ...): calls `fit(X_train, y_train)` to
  learn a mapping, then `predict(X)` to apply it. The `score(X, y)` method predicts and computes a default metric in one step.
- **Pipeline** (Chapter 5): chains a transformer and a predictor so both rules apply automatically inside cross-validation.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

scaler_demo = StandardScaler()
lr_demo = LinearRegression()

X_demo = np.array([[10.0, 80.0, 65.0], [20.0, 90.0, 75.0], [15.0, 85.0, 70.0]])
y_demo = np.array([55.0, 75.0, 65.0])

# Transformer: fit learns mean and std, transform applies them
scaler_demo.fit(X_demo)
print(f"Scaler is a TransformerMixin: {isinstance(scaler_demo, TransformerMixin)}")
print(f"Learned mean_: {scaler_demo.mean_.round(1)}")
print(f"Transformed:\n{scaler_demo.transform(X_demo).round(2)}")

# Predictor: fit learns weights, predict applies them
lr_demo.fit(X_demo, y_demo)
print(f"\nLinearRegression is a BaseEstimator: {isinstance(lr_demo, BaseEstimator)}")
print(f"Learned coef_: {lr_demo.coef_.round(3)}")
print(f"Predictions: {lr_demo.predict(X_demo).round(1)}")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Fit once on training data, then transform/predict on everything else</span><br><br>
The scaler's <code>fit()</code> call computes mean and standard deviation from the training set only. Calling <code>transform()</code> on the test set applies those same statistics. Fitting the scaler on all data (train + test combined) is data leakage: the model has seen test-set statistics before evaluation, so reported scores are optimistic and production performance will be worse.
</div>

## 2. The Dataset: University Analytics

You first encountered `university_analytics.csv` in Part 3 (NumPy) and Parts 8-9 (Pandas).
This notebook reuses the same 2,400 student records to build predictive models.

**Two prediction problems:**

| Target | Type | Task |
| --- | --- | --- |
| `final_score` (0-100, continuous) | Regression | Predict a student's final exam score from early indicators |
| `passed` (True/False, imbalanced) | Classification | Predict whether a student will pass from the same indicators |

**Features used in this notebook** (numerical only):

| Feature | Description |
| --- | --- |
| `study_hours` | Weekly self-reported study hours |
| `attendance_pct` | Lecture attendance percentage |
| `midterm_score` | Midterm exam score (0-100) |

Categorical features (`program`, `gender`, `region`, `has_internet`) require encoding and are deferred to Chapter 5 (Pipelines).


In [ ]:
FEATURES: list[str] = ["study_hours", "attendance_pct", "midterm_score"]
TARGET_REG = "final_score"
TARGET_CLF = "passed"

df_raw = pd.read_csv(DATA_PATH)
logger.info(f"Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# Drop rows where any feature has a missing value
df = df_raw.dropna(subset=FEATURES).reset_index(drop=True)
n_dropped = len(df_raw) - len(df)
logger.info(f"Dropped {n_dropped} rows with missing features | Remaining: {len(df):,} rows")

df[FEATURES + [TARGET_REG, TARGET_CLF]].describe().round(2)

In [ ]:
scatter_df = df[[TARGET_REG] + FEATURES].copy()
scatter_df["study_group"] = pd.cut(df["study_hours"], bins=3, labels=["low", "medium", "high"]).astype(
    str
)  # lets-plot requires plain str dtype, not pandas StringDtype

(
    ggplot(scatter_df, aes(x="midterm_score", y="final_score", color="study_group"))
    + geom_point(alpha=0.4, size=2)
    + scale_color_manual(values=pro_colors[:3])
    + labs(
        title="Midterm score vs Final score by study intensity",
        subtitle="Students who study more tend to score higher, but there is substantial spread",
        x="Midterm score",
        y="Final score",
        color="Study hours",
    )
    + modern_theme(grid=True)
    + ggsize(620, 380)
)

In [ ]:
X = df[FEATURES].to_numpy()
y_reg = df[TARGET_REG].to_numpy()
y_clf = df[TARGET_CLF].astype(int).to_numpy()

logger.info(f"Feature matrix shape: {X.shape}")
logger.info(f"Regression target   : range [{y_reg.min():.0f}, {y_reg.max():.0f}], mean {y_reg.mean():.1f}")
logger.info(f"Classification target: {y_clf.mean():.1%} positive (passed=1) | highly imbalanced")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-pencil-fill"></i> Activity 1: Explore feature correlations</span><br><br>
Compute the Pearson correlation between each of the three features and <code>final_score</code>. Which feature is most predictive? Then create a scatter plot of <code>attendance_pct</code> vs <code>final_score</code> and compare it visually to the midterm plot above. What does the weaker correlation look like in the chart?
</div>

In [ ]:
# Activity 1: your code here
...

## 3. Preprocessing: Feature Scaling

Most sklearn estimators expect features on a comparable scale.
`LinearRegression` converges correctly without scaling (OLS has a closed-form solution),
but `LogisticRegression`, KNN, SVM, and neural networks all require it.
Scaling now builds the habit before you reach those models.

**StandardScaler** subtracts the mean and divides by the standard deviation, producing zero-mean unit-variance features.
Use it as the default for most models.

**MinMaxScaler** maps every feature to [0, 1] by subtracting the minimum and dividing by the range.
Prefer it when the bounded range is meaningful (e.g., image pixel values) or the algorithm requires it.


In [ ]:
# Fit on raw X to show the transformation (we will fit properly on X_train in Section 4)
scaler_ss = StandardScaler()
X_scaled_ss = scaler_ss.fit_transform(X)

print(f"{'Feature':<20} {'Before mean':>12} {'Before std':>11} {'After mean':>11} {'After std':>10}")
print("-" * 68)
for j, name in enumerate(FEATURES):
    print(
        f"{name:<20} {X[:, j].mean():>12.2f} {X[:, j].std():>11.2f}"
        f" {X_scaled_ss[:, j].mean():>11.6f} {X_scaled_ss[:, j].std():>10.2f}"
    )

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-x-circle-fill"></i> Common Mistake: Fitting the scaler before splitting (data leakage)</span><br><br>
The code above calls <code>fit_transform(X)</code> on the full dataset just to demonstrate the transformation.
In a real workflow, this is data leakage: the scaler has seen test-set values when computing mean and std,
so the test set is no longer truly unseen.
<br><br>
The correct order (shown in Section 4) is: split first, then fit the scaler on X_train only, then transform both splits.
<pre>
scaler.fit(X_train)        # learns from training data only
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)   # uses training statistics on test data
</pre>
</div>

In [ ]:
scaler_mm = MinMaxScaler()
X_scaled_mm = scaler_mm.fit_transform(X)

print(f"{'Feature':<20} {'SS min':>8} {'SS max':>8} {'MM min':>8} {'MM max':>8}")
print("-" * 56)
for j, name in enumerate(FEATURES):
    print(
        f"{name:<20} {X_scaled_ss[:, j].min():>8.3f} {X_scaled_ss[:, j].max():>8.3f}"
        f" {X_scaled_mm[:, j].min():>8.3f} {X_scaled_mm[:, j].max():>8.3f}"
    )
print()
print("StandardScaler: zero mean, unbounded range (negative values are expected)")
print("MinMaxScaler  : bounded [0, 1], but extreme values compress the rest")

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-pencil-fill"></i> Activity 2: The double-scaling trap</span><br><br>
Apply <code>StandardScaler</code> twice: fit and transform on <code>X</code>,
then fit and transform again on the result.
What are the mean and standard deviation of the doubly-scaled output?
Explain why the answer is not surprising, and what this tells you about idempotency of linear transformations.
</div>

In [ ]:
# Activity 2: your code here
...

## 4. Splitting the Data

From Chapter 3 Section 3: the training set teaches the model, the test set evaluates it.
The test set must not influence any fitting decision, including fitting the scaler.

`train_test_split` shuffles and splits in one call.
For classification with an imbalanced target, pass `stratify=y` to preserve the class ratio in both splits.


In [ ]:
# Regression: predict final_score
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.20, random_state=42)
logger.info(f"Regression  | train: {len(X_train):,}  test: {len(X_test):,}")

# Scale AFTER splitting (fit only on training data)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train mean (should be ~0.0): {X_train_s.mean(axis=0).round(6)}")
print(f"Test  mean (will not be 0) : {X_test_s.mean(axis=0).round(3)}")

In [ ]:
# Classification: predict passed (96% positive, highly imbalanced)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, y_clf, test_size=0.20, random_state=42, stratify=y_clf)
logger.info(f"Classification | train: {len(Xc_train):,}  test: {len(Xc_test):,}")
logger.info(f"Train positive rate: {yc_train.mean():.1%} | Test positive rate: {yc_test.mean():.1%}")

scaler_c = StandardScaler()
Xc_train_s = scaler_c.fit_transform(Xc_train)
Xc_test_s = scaler_c.transform(Xc_test)

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-x-circle-fill"></i> Common Mistake: Using accuracy as the only metric for an imbalanced target</span><br><br>
The <code>passed</code> column is 96% positive. A classifier that always predicts <em>pass</em>
achieves 96% accuracy without learning anything.
<br><br>
Always use <code>classification_report</code> (precision, recall, F1 per class) instead of accuracy alone
when classes are imbalanced. The dummy baseline in Section 5 will demonstrate this directly.
</div>

## 5. Baseline First

From Chapter 3 Section 5: every model must beat a trivially simple baseline.

- `DummyRegressor(strategy="mean")` always predicts the training-set mean. R2 = 0 by definition.
- `DummyClassifier(strategy="most_frequent")` always predicts the majority class.

If your real model barely outperforms these, you have a problem: wrong features, wrong objective, or a bug.


In [ ]:
dummy_reg = DummyRegressor(strategy="mean")
dummy_reg.fit(X_train_s, y_train)

y_pred_dummy = dummy_reg.predict(X_test_s)
baseline_mae = mean_absolute_error(y_test, y_pred_dummy)
baseline_rmse = root_mean_squared_error(y_test, y_pred_dummy)
baseline_r2 = r2_score(y_test, y_pred_dummy)

logger.info(f"DummyRegressor | MAE={baseline_mae:.2f} | RMSE={baseline_rmse:.2f} | R2={baseline_r2:.3f}")
logger.info(f"Always predicts: {dummy_reg.predict([[0, 0, 0]])[0]:.1f} (training mean)")

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(Xc_train_s, yc_train)

print("DummyClassifier (always predicts 'pass'):")
print(classification_report(yc_test, dummy_clf.predict(Xc_test_s), target_names=["fail", "pass"]))
print("Accuracy looks high (96%) but recall for 'fail' is 0%. The model learned nothing.")

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Baseline first: always</span><br><br>
A baseline is not a starting point to improve from: it is the floor below which your model must not fall.
If your model does not beat the dummy, stop. The problem is upstream of the algorithm:
wrong features, wrong target, data quality issues, or a metric that doesn't match the business goal.
<br><br>
Log both baseline and model metrics side-by-side so the comparison is explicit in every experiment run.
</div>

## 6. Simple Models and Evaluation

With a baseline established, fit `LinearRegression` for regression and `LogisticRegression` for classification.
Type-annotated helper functions log results with loguru and return a metrics dict for later comparison.


In [ ]:
def evaluate_regressor(
    model: Any,
    X_test: np.ndarray,
    y_test: np.ndarray,
    label: str = "model",
) -> dict[str, float]:
    y_pred = model.predict(X_test)
    metrics: dict[str, float] = {
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
        "r2": r2_score(y_test, y_pred),
    }
    logger.info(f"{label:<22} | MAE={metrics['mae']:.2f} | RMSE={metrics['rmse']:.2f} | R2={metrics['r2']:.3f}")
    return metrics


def evaluate_classifier(
    model: Any,
    X_test: np.ndarray,
    y_test: np.ndarray,
    label: str = "model",
    target_names: list[str] | None = None,
) -> None:
    y_pred = model.predict(X_test)
    logger.info(f"Classification report for {label}:")
    print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_s, y_train)

dummy_metrics = evaluate_regressor(dummy_reg, X_test_s, y_test, "DummyRegressor")
lin_metrics = evaluate_regressor(lin_reg, X_test_s, y_test, "LinearRegression")

print()
print("Feature weights (standardised scale):")
for name, coef in zip(FEATURES, lin_reg.coef_, strict=True):
    print(f"  {name:<20} {coef:>+.3f}")
print(f"  {'intercept':<20} {lin_reg.intercept_:>+.3f}")

In [ ]:
y_pred_lin = lin_reg.predict(X_test_s)
resid_df = pd.DataFrame({"predicted": y_pred_lin, "residual": y_test - y_pred_lin})

(
    ggplot(resid_df, aes(x="predicted", y="residual"))
    + geom_point(alpha=0.35, color=pro_colors[0], size=2)
    + geom_hline(yintercept=0, color=pro_colors[1], size=1.0, linetype="dashed")
    + labs(
        title="Residuals vs Fitted: LinearRegression on final_score",
        subtitle="Random scatter around zero suggests no systematic bias; wide spread points to missing predictors",
        x="Predicted final score",
        y="Residual (actual - predicted)",
    )
    + modern_theme(grid=True)
    + ggsize(640, 360)
)

In [ ]:
# class_weight='balanced' up-weights the minority class (fail) to counter the 96/4 imbalance
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg.fit(Xc_train_s, yc_train)

evaluate_classifier(dummy_clf, Xc_test_s, yc_test, "DummyClassifier", ["fail", "pass"])
evaluate_classifier(log_reg, Xc_test_s, yc_test, "LogisticRegression", ["fail", "pass"])

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-pencil-fill"></i> Activity 3: Add Ridge regression to the comparison</span><br><br>
<code>Ridge</code> regression adds L2 regularisation to LinearRegression.
Import <code>Ridge</code> from <code>sklearn.linear_model</code>, fit it with <code>alpha=1.0</code>,
and evaluate it using <code>evaluate_regressor</code>.
Does Ridge outperform plain LinearRegression on this dataset?
Why might regularisation help or not help here?
</div>

In [ ]:
# Activity 3: your code here
...

## 7. Cross-Validation

A single train/test split gives one score that may be lucky or unlucky depending on which rows ended up in each set.
Cross-validation runs k independent train/val splits and returns a distribution of scores.

From Chapter 3: the mean and standard deviation of cross-validation scores together characterise model reliability.
A low mean with low std means consistently poor; low mean with high std means noisy data or too little of it.

**Note:** In these examples we pass pre-scaled data directly to `cross_val_score`.
This is a minor leakage because the scaler was fit on the full training set before CV folds are formed.
Chapter 5 (Pipelines) eliminates this by embedding the scaler inside the Pipeline object,
which then fits correctly within each fold.


In [ ]:
# 5-fold CV for regression (R2 scoring)
cv_scores_reg = cross_val_score(
    LinearRegression(),
    X_train_s,
    y_train,
    cv=5,
    scoring="r2",
)
logger.info(
    f"LinearRegression 5-fold CV R2: {cv_scores_reg.round(3)} | "
    f"mean={cv_scores_reg.mean():.3f} +/- {cv_scores_reg.std():.3f}"
)

In [ ]:
# StratifiedKFold preserves the class ratio (96/4) in every fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_clf = cross_val_score(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    Xc_train_s,
    yc_train,
    cv=skf,
    scoring="f1_macro",
)
logger.info(
    f"LogisticRegression 5-fold stratified CV F1-macro: {cv_scores_clf.round(3)} | "
    f"mean={cv_scores_clf.mean():.3f} +/- {cv_scores_clf.std():.3f}"
)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Use StratifiedKFold for classification whenever classes are imbalanced</span><br><br>
Plain <code>KFold</code> splits rows randomly. On a 96/4 imbalance, some folds may contain no failure cases at all,
making the validation metric meaningless.
<code>StratifiedKFold</code> preserves the class ratio in every fold, ensuring each split
is a representative sample of the full dataset.
</div>

## 8. Learning Curves

From Chapter 3 Section 5: a learning curve plots training score and validation score
against the number of training examples.

- **Both scores low and close together:** underfitting. Add features or use a more complex model.
- **Large gap, training much higher than validation:** overfitting. Regularise or add more data.
- **Both scores high and close together:** good fit.
- **Validation still rising at the right edge:** data-limited. More data should help.

`sklearn.model_selection.learning_curve` computes these curves with built-in cross-validation.


In [ ]:
train_sizes_abs, train_scores, val_scores = learning_curve(
    LinearRegression(),
    X_train_s,
    y_train,
    train_sizes=np.linspace(0.10, 1.0, 10),
    cv=5,
    scoring="r2",
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

logger.info(f"Training R2 at full size : {train_mean[-1]:.3f} +/- {train_std[-1]:.3f}")
logger.info(f"Validation R2 at full size: {val_mean[-1]:.3f} +/- {val_std[-1]:.3f}")
logger.info("Both scores are close and both are low: a textbook underfitting pattern")

In [ ]:
lc_df = pd.DataFrame(
    {
        "n": np.tile(train_sizes_abs, 2),
        "score": np.concatenate([train_mean, val_mean]),
        "lo": np.concatenate([train_mean - train_std, val_mean - val_std]),
        "hi": np.concatenate([train_mean + train_std, val_mean + val_std]),
        "split": ["Training"] * len(train_sizes_abs) + ["Validation"] * len(train_sizes_abs),
    }
)

(
    ggplot(lc_df, aes(x="n", y="score", color="split", fill="split"))
    + geom_ribbon(aes(ymin="lo", ymax="hi"), alpha=0.12)
    + geom_line(size=1.5)
    + scale_color_manual(values=[pro_colors[0], pro_colors[1]])
    + scale_fill_manual(values=[pro_colors[0], pro_colors[1]])
    + labs(
        title="Learning curve: LinearRegression on final_score",
        subtitle=(
            "Training and validation R2 converge near 0.10, with no large gap. "
            "This is underfitting: three features are not enough to fully explain final score."
        ),
        x="Training set size",
        y="R² score",
        color="Split",
        fill="Split",
    )
    + modern_theme(grid=True)
    + ggsize(660, 380)
)

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Plot learning curves before reporting any model result</span><br><br>
A single test-set score answers "how good is this model on this data?"
A learning curve answers "what kind of problem does this model have, and what should I do next?"
Underfitting calls for better features. Overfitting calls for more data or regularisation.
Data-limited calls for a different data collection strategy.
<br><br>
Make the learning curve a standard output of every model training run, not an afterthought.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-pencil-fill"></i> Activity 4: Interpret the learning curve for LogisticRegression</span><br><br>
Run <code>learning_curve</code> with <code>LogisticRegression(class_weight="balanced")</code>
on the classification problem (use <code>scoring="f1_macro"</code>).
Plot the result using the same lets-plot template.
Which of the four Chapter 3 patterns does the curve show?
What does that tell you about what to try next to improve classification performance?
</div>

In [ ]:
# Activity 4: your code here
...

## 9. Model Persistence

From Chapter 3 Section 2 (the 9-stage workflow): deployment requires a serialised model object.
`joblib` is the standard tool for sklearn models: it handles NumPy arrays inside the model efficiently.

Save the model and scaler together as a bundle so that inference code cannot accidentally
apply the wrong scaler (or no scaler) to new data.


In [ ]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "linreg_final_score.joblib"

bundle = {"model": lin_reg, "scaler": scaler, "features": FEATURES}
joblib.dump(bundle, MODEL_PATH)
logger.success(f"Model saved to {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1024:.1f} KB)")

# Reload and verify
loaded = joblib.load(MODEL_PATH)
X_new = np.array([[15.0, 85.0, 72.0]])  # study_hours, attendance_pct, midterm_score
X_new_s = loaded["scaler"].transform(X_new)
pred = loaded["model"].predict(X_new_s)[0]
logger.info(f"Loaded model prediction for {dict(zip(FEATURES, X_new[0], strict=False))}: {pred:.1f}")

# Tidy up
MODEL_PATH.unlink()
MODEL_DIR.rmdir()

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Bundle the model and all preprocessing objects together</span><br><br>
Saving only the fitted model and forgetting the scaler is one of the most common ML bugs in production.
At inference time, raw input data must pass through exactly the same preprocessing steps
that were applied during training. Saving them as a single dict means they travel together
and the version loaded is always consistent with the version trained.
</div>

## Capstone: End-to-End Training Function

Combine everything into a single, type-annotated, logged function that runs the full sklearn workflow.
This is the pattern you will use in every Chapter 5 and Part 5 notebook.


In [ ]:
def train_and_evaluate(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    task: str,
    model_dir: Path = Path("models"),
) -> dict[str, float]:
    """Run the full sklearn workflow for one task.

    Steps: clean -> split -> scale -> baseline -> fit -> CV -> save.

    Args:
        df: Source dataframe. Rows with NaN in features are dropped internally.
        features: Column names used as predictors.
        target: Column name of the response variable.
        task: Either 'regression' or 'classification'.
        model_dir: Directory to write the serialised model bundle.

    Returns:
        Dict with cv_mean and cv_std for the primary scoring metric.
    """
    logger.info(f"[train_and_evaluate] task={task} | target={target} | features={features}")

    X = df.dropna(subset=features)[features].to_numpy()
    y_raw = df.dropna(subset=features)[target]
    y: np.ndarray = y_raw.astype(int).to_numpy() if task == "classification" else y_raw.to_numpy()

    stratify = y if task == "classification" else None
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=stratify)

    sc = StandardScaler()
    X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)

    if task == "regression":
        baseline: Any = DummyRegressor(strategy="mean")
        model: Any = LinearRegression()
        scoring = "r2"
    else:
        baseline = DummyClassifier(strategy="most_frequent")
        model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
        scoring = "f1_macro"

    baseline.fit(X_tr_s, y_tr)
    model.fit(X_tr_s, y_tr)

    if task == "regression":
        evaluate_regressor(baseline, X_te_s, y_te, "DummyBaseline")
        evaluate_regressor(model, X_te_s, y_te, type(model).__name__)
    else:
        evaluate_classifier(baseline, X_te_s, y_te, "DummyBaseline", ["fail", "pass"])
        evaluate_classifier(model, X_te_s, y_te, type(model).__name__, ["fail", "pass"])

    cv_kfold = StratifiedKFold(5, shuffle=True, random_state=42) if task == "classification" else 5
    cv = cross_val_score(model, X_tr_s, y_tr, cv=cv_kfold, scoring=scoring)
    logger.info(f"5-fold CV {scoring}: {cv.mean():.3f} +/- {cv.std():.3f}")

    model_dir.mkdir(exist_ok=True)
    path = model_dir / f"{task}_{target}.joblib"
    joblib.dump({"model": model, "scaler": sc, "features": features}, path)
    logger.success(f"Saved to {path}")

    return {"cv_mean": float(cv.mean()), "cv_std": float(cv.std())}

In [ ]:
MODEL_OUT = Path("models")

reg_results = train_and_evaluate(df, FEATURES, TARGET_REG, "regression", MODEL_OUT)
clf_results = train_and_evaluate(df, FEATURES, TARGET_CLF, "classification", MODEL_OUT)

print("\nSummary:")
print(f"  Regression  CV R2  : {reg_results['cv_mean']:.3f} +/- {reg_results['cv_std']:.3f}")
print(f"  Classification CV F1: {clf_results['cv_mean']:.3f} +/- {clf_results['cv_std']:.3f}")

# Tidy up
import shutil

shutil.rmtree(MODEL_OUT, ignore_errors=True)

## Further Reading

| Resource | What to read |
| --- | --- |
| [sklearn User Guide: Preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html) | Full reference for scalers, encoders, and feature extraction |
| [sklearn User Guide: Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) | All CV strategies and scoring options |
| [sklearn User Guide: Learning curves](https://scikit-learn.org/stable/modules/learning_curve.html) | Validation curves and the full diagnosis toolkit |
| [sklearn User Guide: Model persistence](https://scikit-learn.org/stable/model_persistence.html) | joblib vs pickle, versioning, security considerations |
| Geron, *Hands-On ML with Scikit-Learn* Ch. 2 | The end-to-end ML project walk-through that aligns with Chapter 3 |


## Summary

| Concept | One-line takeaway |
| --- | --- |
| **sklearn API contract** | fit() on training data; predict() / transform() on any split |
| **StandardScaler** | Zero-mean, unit-variance; fit on X_train, transform X_train and X_test separately |
| **MinMaxScaler** | Bounded [0, 1]; use when the bounded range is semantically meaningful |
| **train_test_split** | Use stratify=y for classification on imbalanced targets |
| **DummyRegressor / DummyClassifier** | Set the floor; any real model must beat them |
| **LinearRegression** | Closed-form OLS; coef_ shows feature importance on standardised scale |
| **LogisticRegression** | Use class_weight='balanced' for imbalanced classification |
| **classification_report** | Always use it instead of accuracy alone for imbalanced targets |
| **cross_val_score** | Returns a distribution of scores; report mean and std |
| **StratifiedKFold** | Preserves class ratio in every fold; required for imbalanced classification |
| **learning_curve** | Diagnose underfitting (both scores low) vs overfitting (large gap) |
| **joblib** | Bundle model + scaler + feature names into one object before saving |

**Next:** Chapter 5 (Part 23) fixes the minor leakage in this notebook by embedding the scaler inside a `Pipeline`,
adds categorical feature encoding via `ColumnTransformer`, and introduces Optuna for hyperparameter search.
